# 🏠😊💷💶 Keeping Up With The Jones:- Hypothesis Testings

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

## Set Up directories and Import Necessary Libraries

In [2]:
# System and OS related tasks
import sys
import os

# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [3]:
# Get the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pingouin as pg

sns.set_style('whitegrid') # sets a white background with grid lines 

In [4]:
# Get data:

filename = "2a_df_regional_wellbeing_6countries_mainland.csv"

df_regional_wellbeing_6countries_mainland = pd.read_csv(
    f"{processed_dir}/{filename}",
    )

df_regional_wellbeing_6countries_mainland.head(3)

,xls_row_id,country,region,region_code,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,...,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index,country_iso3,north_south
0,111,France,Île-de-France,FR1,35629.0,68.7,7.9,1.0,84.9,84.1,...,6.9,76.0,94.2,12.7,6.7,43.4,55.0,88.8,FRA,Other
1,112,France,Centre-Val de Loire,FRB,30823.6,70.7,5.8,0.8,82.6,86.5,...,8.3,75.0,87.4,8.9,6.4,47.9,53.2,88.5,FRA,Other
2,113,France,Bourgogne-Franche-Comté,FRC,30839.7,68.9,6.6,0.6,82.3,82.0,...,8.4,77.1,83.9,9.8,6.7,41.5,45.8,95.4,FRA,Other


---

# 1.0 Predictive Modelling

📌 In the previous notebook of hypothesis testings, we have looked at the familiar "North-South Divide" narrative across the dimensions of wealth, health and happiness. Hypotheses 1 and 3 confirmed that Northern region tend to have lower disposable income and life expectancy but that there is no apparent difference in life satisfaction as compared to its Southern neighbours.

📌 When we look at the relationships amongst these dimensions within UK and compared it to the selected 5 European nations, we found that all three relationship showed meaning differences. The most interesting was the income-health relations whereby the UK regions showed a strong positive correlation (r = 0.66) while the chosen European counterparts showed no relation at all (r = -0.08). The income-happiness link is also stronger in the European five (r = 0.39) than the UK (r = 0.18), giving the impression taht money buys more happiness elsewhere than it does in the UK.

📌 In this notebook, we will attempt to create the following predictive models to explore these relationships further:
* Model 1: `MVP` _**"The Baseline"**_ Income predicts happiness for all regions (baseline)
* Model 2: `MVP` _**"The Joneses Comparison"**_ Includes the UK as a variable so as to acertain if UK regrios are systematically happier or unhappier than the European regions on the similar income.
* Model 3: `Coud Have` _**"Health and Happiness"**_ For the UK alone, does health adds anything to the prediction of happiness beyond income. 

---

# 2.0 Predictive Model 1: "The Baseline"

📌 This model aims to test whether income can predict happiness across regions.

## 2.1 Split the Data into Train and Test

In [5]:
m1_want_col_list = ["disposable_income_pc", "life_expectancy", "life_satisfaction_index"]

df_model1 = df_regional_wellbeing_6countries_mainland.copy()

df_model1 = df_regional_wellbeing_6countries_mainland[m1_want_col_list]

df_model1.head(3)

,disposable_income_pc,life_expectancy,life_satisfaction_index
0,35629.0,84.9,6.7
1,30823.6,82.6,6.4
2,30839.7,82.3,6.7


In [7]:
# double check for missing values
print(df_model1.isnull().sum())

disposable_income_pc       0
life_expectancy            0
life_satisfaction_index    0
dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                    df_model1.drop(['life_satisfaction_index'], axis=1), # for the X dependent variables& dropping the target variable
                                    df_model1['life_satisfaction_index'],# add the target valurable
                                    test_size=0.2, # 80/20 split into train and test
                                    random_state=101  # set the seed for the random generator for the splitting process
                                    )

print(
    "* Train set:",
    X_train.shape,
    y_train.shape,
    "\n* Test set:",
    X_test.shape,
    y_test.shape,
)

* Train set: (72, 2) (72,) 
* Test set: (19, 2) (19,)


## 2.2 Build the Pipeline

📌  The `StandardScaler` is used so that features like `life_expectancy` with double digit values will not be overwhelmed by `disposable_income_pc` with large values.

📌  We will rely on the feature selector `Lasso` as it can automatically select features that are useful by setting unused features' coefficient to zero. As we have only 2 features to play with, we will set `alpha` to be very small to keep as many features as necessary.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel

def pipeline_linear_regression():
    """
    Create a pipeline for linear regression.

    This function creates a pipeline that performs feature scaling, feature selection,
    and linear regression modeling.

    Returns:
        pipeline (Pipeline): The pipeline object that performs the specified steps.
    """
    pipeline = Pipeline([
        ("feat_scaler", StandardScaler()),
        ("feat_selector", SelectFromModel(Lasso(alpha=0.01))),  # 0.01 so Lasso keeps as many features as possible
        ("model", LinearRegression())
    ])
    return pipeline

pipeline_linear_regression()

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feat_scaler', ...), ('feat_selector', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"estimator estimator: objectThe base estimator from which the transformer is built.This can be both a fitted (if ``prefit`` is set to True)or a non-fitted estimator. The estimator should have a``feature_importances_`` or ``coef_`` attribute after fitting.Otherwise, the ``importance_getter`` parameter should be used.",Lasso(alpha=0.01)
,"threshold threshold: str or float, default=NoneThe threshold value to use for feature selection. Features whoseabsolute importance value is greater or equal are kept while the othersare discarded. If ""median"" (resp. ""mean""), then the ``threshold`` valueis the median (resp. the mean) of the feature importances. A scalingfactor (e.g., ""1.25*mean"") may also be used. If None and if theestimator has a parameter penalty set to l1, either explicitlyor implicitly (e.g, Lasso), the threshold used is 1e-5.Otherwise, ""mean"" is used by default.",None
,"prefit prefit: bool, default=FalseWhether a prefit model is expected to be passed into the constructordirectly or not.If `True`, `estimator` must be a fitted estimator.If `False`, `estimator` is fitted and updated by calling`fit` and `partial_fit`, respectively.",False
,"norm_order norm_order: non-zero int, inf, -inf, default=1Order of the norm used to filter the vectors of coefficients below``threshold`` in the case where the ``coef_`` attribute of theestimator is of dimension 2.",1


## 2.3 Teach the Machine for Fitting

📌 Get the trained linear regression model's intercept and coefficients using the function.

In [13]:
def linear_model_coefficients(model, columns):
    """
    Print the intercept and coefficients of a linear model.

    Parameters:
    model (object): The trained linear model object.
    columns (array-like): The column names corresponding to the coefficients.

    Returns:
    None
    """
    print(f"* Interception: {model.intercept_}")
    coeff_df = pd.DataFrame(model.coef_, columns, columns=["Coefficient"]).sort_values(
        ["Coefficient"], key=abs, ascending=False
    )
    print("* Coefficients")
    print(coeff_df)

📌 Create and train the pipeline

In [14]:
# Create the pipeling
pipeline = pipeline_linear_regression()

# Train the pipeline
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feat_scaler', ...), ('feat_selector', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"estimator estimator: objectThe base estimator from which the transformer is built.This can be both a fitted (if ``prefit`` is set to True)or a non-fitted estimator. The estimator should have a``feature_importances_`` or ``coef_`` attribute after fitting.Otherwise, the ``importance_getter`` parameter should be used.",Lasso(alpha=0.01)
,"threshold threshold: str or float, default=NoneThe threshold value to use for feature selection. Features whoseabsolute importance value is greater or equal are kept while the othersare discarded. If ""median"" (resp. ""mean""), then the ``threshold`` valueis the median (resp. the mean) of the feature importances. A scalingfactor (e.g., ""1.25*mean"") may also be used. If None and if theestimator has a parameter penalty set to l1, either explicitlyor implicitly (e.g, Lasso), the threshold used is 1e-5.Otherwise, ""mean"" is used by default.",None
,"prefit prefit: bool, default=FalseWhether a prefit model is expected to be passed into the constructordirectly or not.If `True`, `estimator` must be a fitted estimator.If `False`, `estimator` is fitted and updated by calling`fit` and `partial_fit`, respectively.",False
,"norm_order norm_order: non-zero int, inf, -inf, default=1Order of the norm used to filter the vectors of coefficients below``threshold`` in the case where the ``coef_`` attribute of theestimator is of dimension 2.",1


📌 Inspect the trained linear regression model from the pipeline

In [15]:
pipeline['model']

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


📌 Look at which features are selected for the trained linear regression model:

In [16]:
pipeline['feat_selector'].get_support()

array([ True,  True])

 * The `lasso` selector has included both `disposable_income_ps` and `lofe_expectancy`

In [17]:
X_train.columns[pipeline['feat_selector'].get_support()]

Index(['disposable_income_pc', 'life_expectancy'], dtype='str')

📌 Get the intercept and the coefficients using the handy function `linear_model_coefficients`. 

In [18]:
linear_model_coefficients(
    model=pipeline['model'],
    columns=X_train.columns[pipeline['feat_selector'].get_support()])

* Interception: 6.7027777777777775
* Coefficients
                      Coefficient
disposable_income_pc     0.172781
life_expectancy         -0.123007


📌 Look at the predicted min and max `life_satisfaction_index` values from the model. 

In [20]:
print(f"The min predicted life_satisfaction_index: {pipeline.predict(X_test).min()}")
print(f"The min predicted life_satisfaction_index: {pipeline.predict(X_test).max()}")

The min predicted life_satisfaction_index: 6.586444876879412
The min predicted life_satisfaction_index: 6.988909078679335


* whilst the range of min and max looks small, the 6 nation data showed that the `life_satisfaction_index` was between the range of 5.5 to 7.5.

📌 The linear regression model is:

>
>`Happiness`  = 6.7 + (0.17 × `Income`) + (-0.12 × `Health`)
>
or more accurately:
>
>`life_satisfaction_index`  = 6.702778 + (0.172781 × `disposabble_income_pc`) + (-0.123007 × `life_expectancy`)
>

📌🔍📈 What the linear regression model is saying:
* when income and health is zero, the hapiness is 6.7.
  * In reality, this does not make any sense as one cannot be 6.7 (out of 10) satisfied with life when one is not alive.
* the income is positively related to the yield. For every US$1 increase in disposable income, the happiness index would increase by 0.173.
  * this does not go against the common notion that wealth brings happinesses (to a certain extent)
* the health is negatively related to the happiness index such that for every 1 year of life, the happiness index would drop by 0.123.

📌🔍🕵🏻‍♂️ It is perculiar to see that happiness would fall as one gets older. 

# 2.4 Evaluate the Model

---

# 3.0 Predictive Model 2: 

📌

## 3.1 Split the Data into Train and Test

## 3.2 Build the Pipeline

## 3.3 Teach the Machine for Fitting

## 3.4 Evaluate the Model

---

# 4.0 Predictive Model 3??: 

📌

## 4.1 Split the Data into Train and Test

## 4.2 Build the Pipeline

## 4.3 Teach the Machine for Fitting

## 4.4 Evaluate the Model

---

# 5.0 Conclusion, Limitations and Ethical Considerations:

📌

---